# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

# Unit of Analysis + Time Window

## Data Contract

One row represents **one anonymized content page observation**.

Each observation contains measured search intelligence signals for a content page, including:
- search demand
- click efficiency
- ranking position
- engagement behavior
- performance trend

The selected analysis period is the available historical dataset snapshot used for development. The model will use this historical information to rank content pages by optimization opportunity.

The reason for choosing a page-level unit is that the final business action is page-level: deciding which content assets should be improved first.

This contract separates:
- features available before a decision is made
- labels or proxy outcomes used for evaluation

to ensure the model remains realistic and avoids information leakage.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

# Feature, Label, Context, and Excluded Fields

## Feature Fields

The model can use the following observed signals:

| Field | Role |
|---|---|
| search_volume | Measures available search demand |
| ctr | Measures how efficiently impressions generate clicks |
| avg_position | Measures current ranking visibility |
| engagement_rate | Measures user interaction quality |
| trend_pct | Measures recent performance direction |

---

## Label / Ranking Target

The business objective is:

"Identify content pages with the highest opportunity for optimization."

Since the dataset does not contain a direct future business outcome after optimization, the ranking objective will use observed performance indicators as proxy signals.

The proxy considers:
- declining trends
- ranking position opportunity
- available search demand
- engagement signals

The output will be a prioritized list rather than a binary prediction.

---

## Context Fields

| Field | Purpose |
|---|---|
| content_id | Identifies each content page |
| client_id | Provides grouping context |
| trend_direction | Provides categorical trend information |

---

## Excluded Fields

| Field | Why excluded |
|---|---|
| Future optimization results | Not available during prediction |
| Post-refresh metrics | Would create leakage |
| Client identifiers as predictive features | Could cause the model to memorize clients instead of learning general patterns |

The goal is to build a model that discovers transferable search opportunity patterns.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

# Data Contract Verification

Each assumption in the contract is validated using measurable checks.

The following checks verify:
1. Dataset grain
2. Dataset size and coverage
3. Missing values and data quality

In [8]:
# Verify one row = one content page
import pandas as pd
df = pd.read_csv("/content/content_refresh_anonymized.csv")
rows = len(df)
unique_pages = df["content_id"].nunique()

print("Total rows:", rows)
print("Unique content pages:", unique_pages)

if rows == unique_pages:
    print("Verified: one row represents one content page.")
else:
    print("Multiple observations exist per content page.")

Total rows: 30000
Unique content pages: 30000
Verified: one row represents one content page.


In [9]:
# Dataset size verification

print("Dataset Shape:", df.shape)

display(df.head())

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [10]:
# Missing value verification

missing_values = df.isnull().sum()

display(
    missing_values[missing_values > 0]
)

,0
search_volume,2468
competition,2468
competition_level,2610
cpc,2468
main_intent,2374
word_count,7699
char_count,7699
provider_used,21438
model_used,5733
word_count_tier,7699


## Grain Verification Result

The validation confirms:

- Total rows: 30,000
- Unique content pages: 30,000

Since the number of rows equals the number of unique `content_id` values, the dataset follows the expected contract:

**One row represents one unique anonymized content page observation.**

This matches the business decision unit because content optimization decisions are made at the individual page level.

## Dataset Coverage Result

The dataset contains:

- 30,000 content page observations
- 44 available fields

The available columns provide multiple categories of search intelligence signals:

- Search demand signals (`search_volume`, `cpc`, competition)
- Ranking signals (`avg_position`, position tiers)
- Engagement signals (`ctr`, `engagement_rate`, `scroll_rate`)
- Content characteristics (`word_count`, `char_count`, content type)
- Trend signals (`trend_direction`, `trend_pct`)

This provides enough information to build a ranking system for content opportunity prioritization.

## Missing Value Analysis

The dataset contains missing values across several observed fields.

The highest missingness appears in:

| Field | Missing Count |
|---|---:|
| provider_used | 21,438 |
| model_used | 5,733 |
| word_count | 7,699 |
| char_count | 7,699 |
| word_count_tier | 7,699 |
| char_count_tier | 7,699 |

These missing values reveal important data limitations.

Possible interpretations:

- Some AI-related fields may only exist for content generated or processed through specific providers.
- Content metadata fields may be unavailable for some pages.
- Missing values should be handled carefully during feature engineering rather than removed blindly.

For the initial ranking model, missingness itself may become a useful signal because the absence of metadata can represent a real data availability pattern.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# Data Limits

This dataset provides strong search intelligence signals, but several limitations must be considered.

## 1. Missing metadata

Some fields have incomplete coverage. For example, AI-related metadata fields such as provider/model information are missing for many observations. Therefore, these fields should not automatically be treated as universally available prediction signals.

## 2. No direct optimization outcome

The dataset shows observed content performance but does not contain the guaranteed future result after a content refresh. Therefore, the model can rank opportunities but cannot claim causal improvement.

## 3. External ranking factors

Search performance can be influenced by external factors not captured in this dataset:

- competitor changes
- search algorithm updates
- seasonality
- market changes

## 4. Client-level variation

Different clients may have different content strategies and audiences. Client identifiers should be treated as context rather than predictive shortcuts.

## Conclusion

The model output should be interpreted as a decision-support ranking system that helps teams prioritize content opportunities, not as a guaranteed prediction of future rankings.

## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.